# Create PaymentsDepositsModel (Payments & Deposits subset)

Builds a **Direct Lake** semantic model named **PaymentsDepositsModel** over the **Payments** subset of the same `gold` tables `IQModel` uses, then adds relationships, KPI measures, hierarchies, and the **full refresh** Direct Lake needs.

**Tables:** dim_customer, dim_account, dim_merchant, dim_product, dim_branch, dim_date, fact_transaction, fact_fraud_alert. *(fact_loan / fact_delinquency are Lending — excluded.)*

**Before you run (order matters):**
1. Open this notebook in the **same workspace** as `IQModel` (here: `YOUR_WORKSPACE_NAME`, tenant YOUR_TENANT).
2. In the **config cell**, set `LAKEHOUSE` to the lakehouse that holds the `gold` tables (the one `IQModel` is built over) and confirm `SCHEMA`.
3. **Run the first code cell (`%pip install semantic-link-labs`) by itself FIRST** and wait for **`Successfully installed … semantic-link-labs-…`** (`sempy_labs` is not preinstalled in Fabric).
4. **Then** run the remaining cells top to bottom. `overwrite=True` makes re-runs safe.


In [ ]:
%pip install semantic-link-labs

In [ ]:
# ---- config: set these to match where IQModel's gold tables live ----
LAKEHOUSE = "NorthwindLakehouse"   # <-- the lakehouse holding the gold tables (same source as IQModel)
SCHEMA    = "gold"
MODEL     = "PaymentsDepositsModel"

# Payments & Deposits tables (subset of IQModel)
TABLES = ["dim_customer", "dim_account", "dim_merchant", "dim_product",
          "dim_branch", "dim_date", "fact_transaction", "fact_fraud_alert"]

# relationships: (from_table, from_col, to_table, to_col) - all Many -> One, single direction
RELS = [
    ("fact_transaction", "customerKey", "dim_customer", "customerKey"),
    ("fact_transaction", "merchantId",  "dim_merchant", "merchantId"),
    ("fact_transaction", "accountId",   "dim_account",  "accountId"),
    ("fact_transaction", "dateKey",     "dim_date",     "dateKey"),
    ("fact_fraud_alert", "customerKey", "dim_customer", "customerKey"),
    ("dim_account",      "productId",   "dim_product",  "productId"),
    ("dim_account",      "branchId",    "dim_branch",   "branchId"),
]

# measures: (home_table, name, DAX, display_folder)
MEASURES = [
    ("fact_transaction", "Transaction Count",  "COUNTROWS(fact_transaction)", "Volumes"),
    ("fact_transaction", "Transaction Amount", "SUM(fact_transaction[amount])", "Volumes"),
    ("fact_transaction", "Fee Income",         "SUM(fact_transaction[amount]) * 0.015", "Volumes"),
    ("fact_transaction", "Fraud Txn Count",    "CALCULATE(COUNTROWS(fact_transaction), fact_transaction[isFraud] = TRUE())", "Fraud"),
    ("fact_transaction", "Fraud Loss Amount",  "CALCULATE(SUM(fact_transaction[amount]), fact_transaction[isFraud] = TRUE())", "Fraud"),
    ("fact_transaction", "Fraud Loss Rate",    "DIVIDE([Fraud Loss Amount], [Transaction Amount])", "Fraud"),
    ("dim_account",      "Deposit Balance",    "SUM(dim_account[balance])", "Balances"),
    ("fact_fraud_alert", "Open Fraud Alerts",  'CALCULATE(COUNTROWS(fact_fraud_alert), fact_fraud_alert[status] = "Open")', "Fraud"),
    ("fact_fraud_alert", "Customers With Fraud Alert", "DISTINCTCOUNT(fact_fraud_alert[customerKey])", "Fraud"),
]

# hierarchies: table -> (name, [levels])
HIERS = {
    "dim_date":    ("Calendar",          ["year", "quarter", "month"]),
    "dim_branch":  ("Geography",         ["region", "name"]),
    "dim_product": ("Product Hierarchy", ["family", "name"]),
}
print("config set for", MODEL)

In [ ]:
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

# 1) create the Direct Lake model over the Payments subset of gold
labs.directlake.generate_direct_lake_semantic_model(
    dataset=MODEL,
    tables={t: f"{SCHEMA}.{t}" for t in TABLES},
    source=LAKEHOUSE,
    source_type="Lakehouse",
    overwrite=True,
    refresh=False,
)

# 2) relationships + measures + hierarchies
with connect_semantic_model(dataset=MODEL, readonly=False) as tom:
    for ft, fc, tt, tc in RELS:
        try:
            tom.add_relationship(from_table=ft, from_column=fc, to_table=tt, to_column=tc,
                                 from_cardinality="Many", to_cardinality="One")
        except Exception as e:
            print("rel skip", ft, fc, ":", e)
    for t, nm, dax, folder in MEASURES:
        try:
            tom.add_measure(table_name=t, measure_name=nm, expression=dax, display_folder=folder)
        except Exception as e:
            print("measure skip", nm, ":", e)
    for t, (hn, cols) in HIERS.items():
        try:
            tom.add_hierarchy(table_name=t, hierarchy_name=hn, columns=cols)
        except Exception as e:
            print("hierarchy skip", hn, ":", e)
print("built + enriched", MODEL)

In [ ]:
# 3) Direct Lake REQUIRES a full refresh so relationship indexes build
labs.refresh_semantic_model(dataset=MODEL, refresh_type="full")
print("refreshed", MODEL)

In [ ]:
# 4) verify KPIs
import sempy.fabric as fabric
df = fabric.evaluate_dax(dataset=MODEL, dax_string=
    'EVALUATE ROW("Txns",[Transaction Count], "TxnAmount",[Transaction Amount], '
    '"FraudLossRate",[Fraud Loss Rate], "DepositBal",[Deposit Balance])')
print(df.to_string(index=False))